# detach-stop-gradient-trick — ex1: detach G's output during D-step so grad does not flow into G

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `detach-stop-gradient-trick`. Running the final beacon cell reports progress against the `GAN: detach stop-gradient trick` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: detach stop-gradient trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`detach-stop-gradient-trick`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "detach-stop-gradient-trick"
DD_SUBTOPIC = "GAN: detach stop-gradient trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## GAN: `.detach()` stop-gradient trick — quick refresher

When training the discriminator on a fake sample produced by the generator, you MUST detach the fake before passing it to D:

```python
fake = G(z)
loss_D = bce(D(fake.detach()), zeros)    # backward stops at fake
loss_D.backward()
D_opt.step()
```

**What `.detach()` does.** Returns a new tensor that SHARES storage with the original but has `requires_grad=False` and is treated as a leaf by autograd. Backward through `fake.detach()` stops at that node — no grad flows into G's parameters.

**Why the D-step needs this.** During the D-step you're computing `d(loss_D)/d(D's params)`. If you forgot `.detach()`, autograd would ALSO compute `d(loss_D)/d(G's params)` — and then when you later run the G-step's `.backward()`, G's grads are already populated with the WRONG gradient (the one that would make G easier for D to detect, the opposite of what you want).

**Why the G-step doesn't detach.** In `loss_G = -D(G(z)).log().mean()` you WANT the gradient to flow through D and into G. D's params are frozen-by-convention during the G-step (you only call `G_opt.step()`), so D's weights don't move even though grads accumulate in `D.grad`.

**Cheaper alternative for the D-step.** `with torch.no_grad(): fake = G(z)`. Identical effect, slightly faster because the forward doesn't build the autograd graph at all. `.detach()` is more common in published GAN code because it makes the stop-gradient point explicit at the use site.

### Exercise 1 — detach G's output during D-step so grad does not flow into G

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the role of `.detach()` in the GAN D-step by computing D's loss on G's output and verifying that G's parameters receive ZERO gradient when (and only when) `detach()` is applied.
> Keywords: detach, stop-gradient, autograd, gan
> ```

**KCs targeted:** `detach-stops-backward`, `no-grad-in-d-step`

Implement two functions: `ex1_d_loss_correct(G, D, z, x_real)` and `ex1_d_loss_buggy(G, D, z, x_real)`. The ONLY difference is whether `G(z)` is detached.

`ex1_d_loss_correct`:
  1. `fake = G(z).detach()`
  2. `loss = (D(fake) - D(x_real)).mean()`
  3. `loss.backward()`
  4. Return `loss.item()`.

`ex1_d_loss_buggy` — same body, but WITHOUT `.detach()`:
  1. `fake = G(z)`
  2. `loss = (D(fake) - D(x_real)).mean()`
  3. `loss.backward()`
  4. Return `loss.item()`.

Both functions are called fresh (the caller zeroes grads and then runs ONLY this function — no other backward passes in between).

The test will:
1. Call `ex1_d_loss_correct` and assert every G parameter has `.grad is None` OR `.grad` is exactly zero — backward STOPPED at the detach.
2. Call `ex1_d_loss_buggy` and assert at least one G parameter has a NON-zero `.grad` — backward DID flow into G (the bug we're warning against).

The returned loss values should be approximately equal (detach doesn't change the forward, only the backward graph) — within `1e-5`.

In [ ]:
def ex1_d_loss_correct(G, D, z, x_real):
    fake = G(z).detach()
    loss = (D(fake) - D(x_real)).mean()
    loss.backward()
    return loss.item()

def ex1_d_loss_buggy(G, D, z, x_real):
    fake = G(z)
    loss = (D(fake) - D(x_real)).mean()
    loss.backward()
    return loss.item()


<details><summary>Solution</summary>

```python
def ex1_d_loss_correct(G, D, z, x_real):
    fake = G(z).detach()
    loss = (D(fake) - D(x_real)).mean()
    loss.backward()
    return loss.item()

def ex1_d_loss_buggy(G, D, z, x_real):
    fake = G(z)
    loss = (D(fake) - D(x_real)).mean()
    loss.backward()
    return loss.item()
```

**The forward values match exactly.** `.detach()` doesn't change the VALUE of the tensor — same numbers, same shape, same dtype. It just removes the autograd connection back to G. The two losses are bit-identical in the forward.

**D's gradient is the same either way.** Removing the G→fake→D edge from the autograd graph doesn't change the D→loss gradients — D's path to the loss is unaffected. That's why the test asserts `d_correct[name] == d_buggy[name]` for every D param.

**Why the bug is invisible at runtime.** In the buggy version, `G_opt.zero_grad()` at the start of the next G-step would WIPE the wrong gradient — so the model STILL trains, just with the D-step contributing nothing meaningful to G's update. The symptom is 'GAN doesn't converge'; root cause is subtle.

**Alternative formulations:**
- `with torch.no_grad(): fake = G(z)` — cheaper (forward doesn't build graph) but obscures intent.
- `fake = G(z).detach().requires_grad_(False)` — redundant; `.detach()` already implies no grad.
- `fake = G(z); fake.requires_grad = False` — DOESN'T WORK on non-leaf tensors. Use `.detach()`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()